# Per-Subject DSAR Erasure · 1. PII column registry

The registry is the **config** that drives the whole erasure engine (the equivalent of the DBA-provided PII
column list in the current AWS flow). One row per **PII column**, declaring:

- `table_name` — which table it lives in
- `column_name` — the physical column
- `identifier_role` — how this column is used to **match** a subject: `first_name`, `last_name`, `email`,
  `full_name` (single "First Last" column), or `none` (it's PII to erase but not used for matching, e.g. `pnr`)
- `erase_strategy` — how to erase it once a row matches: `scalar_token`, `json_key`, or `pnr_token`
- `json_tag` — for `json_key` columns, which compiled mask to apply (`name` / `email`), reusing the masking repo's approach
- `is_identifier` — whether this column participates in the WHERE match

**Matching rule (your decision — most conservative):** a subject matches a row only if **every identifier
column registered for that table matches**. So `customer_profile` (first+last+email) requires all three;
`contact_email_only` (email only) requires just email; `booking` (full_name) matches on the full name string;
`loyalty_split` requires first_nm AND last_nm.

## 0. Configuration

In [3]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema",  "allegiant_air_dsar", "2 Schema")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
FQ      = f"{CATALOG}.{SCHEMA}"
print("Schema:", FQ)

Schema: dkushari_uc.allegiant_air_dsar

## 1. Create + populate `pii_column_registry`
Rebuilt each run so it always reflects the config below. Edit these rows (or generate them from governed
tags) to onboard new tables — the engine is fully config-driven and needs no code change per table.

In [5]:
spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.pii_column_registry (
  table_name      STRING,
  column_name     STRING,
  identifier_role STRING,   -- first_name | last_name | email | full_name | none
  is_identifier   BOOLEAN,  -- participates in the WHERE match?
  erase_strategy  STRING,   -- scalar_token | json_key | pnr_token
  json_tag        STRING    -- name | email  (only for json_key)
) COMMENT 'Config: which columns are PII, how to match a subject, and how to erase each column.'
""")

registry = [
    # table,              column,       identifier_role, is_identifier, erase_strategy, json_tag
    ("customer_profile",  "first_name", "first_name", True,  "scalar_token", None),
    ("customer_profile",  "last_name",  "last_name",  True,  "scalar_token", None),
    ("customer_profile",  "email",      "email",      True,  "scalar_token", None),

    ("contact_email_only","email",      "email",      True,  "scalar_token", None),

    ("booking",           "full_name",  "full_name",  True,  "scalar_token", None),
    ("booking",           "pnr",        "none",       False, "pnr_token",    None),

    ("loyalty_split",     "first_nm",   "first_name", True,  "scalar_token", None),
    ("loyalty_split",     "last_nm",    "last_name",  True,  "scalar_token", None),

    # JSON: the payload column is both the match surface (identifier_role=full_name via the embedded name)
    # and the erase target (json_key). We match on the embedded first/last/email inside the JSON.
    ("app_hits_json",     "hit_payload","email",      True,  "json_key",     "name"),
]

def s(v):
    return "NULL" if v is None else "'" + v.replace("'","''") + "'"

vals = ",\n".join(
    f"({s(t)}, {s(c)}, {s(r)}, {str(i).lower()}, {s(e)}, {s(j)})"
    for t,c,r,i,e,j in registry
)
spark.sql(f"INSERT INTO {FQ}.pii_column_registry VALUES\n{vals}")
display(spark.table(f"{FQ}.pii_column_registry"))

table_name,column_name,identifier_role,is_identifier,erase_strategy,json_tag
customer_profile,first_name,first_name,True,scalar_token,None
customer_profile,last_name,last_name,True,scalar_token,None
customer_profile,email,email,True,scalar_token,None
contact_email_only,email,email,True,scalar_token,None
booking,full_name,full_name,True,scalar_token,None
booking,pnr,none,False,pnr_token,None
loyalty_split,first_nm,first_name,True,scalar_token,None
loyalty_split,last_nm,last_name,True,scalar_token,None
app_hits_json,hit_payload,email,True,json_key,name


## 2. Note on the JSON table's matching
For `app_hits_json` the identifiers are **inside** the JSON string. The engine matches a subject when the
payload contains that subject's email (and, for extra safety, first & last name) via `LIKE`/`regexp` on the
string, then applies the compiled `json_key` mask to redact `name` + URL `firstName`/`lastName`/`email` for
**only those matched rows** — the schema and every non-PII value (`appName`, `revenue`) are preserved.

Next: **`02_subject_erasure_engine`**.